# Qwen3.5-0.8B V4: Full attention FIRST → Progressive Delta SECOND

This notebook produces **two distinct models**: full-attention-only (6 replacements, retaining all 18 original DeltaNet layers) and an experimental progressively substituted Delta version (18 possible replacements, selected with rollback). Results and models are saved separately.

**Why:** V3 recorded ~59.4% holdout next-token teacher agreement despite faster prefill and exhibited repeating chat output. Replacing 18 DeltaNet layers before validating the six full attention layers obscured the source of the problem. V4 checks full-attention parity first, then ranks individual Delta layers on TRAIN data, adds them gradually, trains and rolls back groups that fail DEV fidelity.

**Splits:** official WikiText train/validation/test; holdout is sealed until final evaluation. FFNs/norms remain frozen. No CeNN. Text-only. use_cache=False; no streaming performance claims.

**Output:** full-only checkpoint, best accepted Delta checkpoint (or first rejected Delta diagnostic-only checkpoint), statistics ZIP and optional model ZIPs; separate Hugging Face gates for each model.


In [ ]:
#@title 1. Environment and experiment parameters
%pip -q install -U "git+https://github.com/huggingface/transformers.git@main" "accelerate>=1.0" "datasets>=3" "huggingface_hub>=0.34" "safetensors>=0.5"
%pip -q install --no-deps "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@a17f294d0144c2a09c41ace8b9fbea6108af4eaa"
from pathlib import Path
from datetime import datetime, timezone
import os, random, math, json, copy, gc, hashlib, zipfile, re, shutil, time
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import HfApi, login, get_token
from IPython.display import display, FileLink
from tinycenn_lm import qwen35_window_anchor_v4 as wa

MODEL_ID="Qwen/Qwen3.5-0.8B"
SEED=20261018
MAX_TOKENS=192 #@param {type:"integer"}
DELTA_WINDOW=128 #@param {type:"integer"}
FULL_WINDOW=512 #@param {type:"integer"}
ANCHORS=16 #@param {type:"integer"}
CHUNK=64 #@param {type:"integer"}
TRAIN_EXAMPLES=320 #@param {type:"integer"}
DEV_EXAMPLES=64 #@param {type:"integer"}
HOLDOUT_EXAMPLES=96 #@param {type:"integer"}
RANKING_EXAMPLES=12 #@param {type:"integer"}
DELTA_STEPS=100 #@param {type:"integer"}
GRAD_ACCUM=2 #@param {type:"integer"}
LORA_RANK=12 #@param {type:"integer"}
LR_START=0.00035 #@param {type:"number"}
EVAL_EVERY=25
DEV_MAX_DROP=0.025
AUTO_DOWNLOAD_STATS=True #@param {type:"boolean"}
AUTO_DOWNLOAD_MODELS=False #@param {type:"boolean"}
UPLOAD_IF_QUALIFIED=True #@param {type:"boolean"}
HF_PRIVATE=True #@param {type:"boolean"}
FULL_HF_NAME="qwen35-08b-v4-full6-only" #@param {type:"string"}
DELTA_HF_NAME="qwen35-08b-v4-progressive-delta" #@param {type:"string"}
BASE=Path("/content/Qwen35_V4_FullFirst_ProgressiveDelta")
REPORTS=BASE/"statistics"
FULL_EXPORT=BASE/"full6_only_model"
DELTA_EXPORT=BASE/"progressive_delta_model"
DIAGNOSTIC_EXPORT=BASE/"rejected_delta_diagnostic_model"
for path in (BASE,REPORTS,FULL_EXPORT):path.mkdir(parents=True,exist_ok=True)
assert 8<=MAX_TOKENS<=512 and FULL_WINDOW>=MAX_TOKENS
assert 0<=ANCHORS<=DELTA_WINDOW and DELTA_WINDOW>=16
assert TRAIN_EXAMPLES>=RANKING_EXAMPLES>=4
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
assert torch.cuda.is_available(),"Enable GPU in Colab runtime settings"
torch.cuda.manual_seed_all(SEED)
device=torch.device("cuda",torch.cuda.current_device())
ATTENTION_SETTINGS={"delta_window":DELTA_WINDOW,"full_window":FULL_WINDOW,
                    "anchors":ANCHORS,"chunk_size":CHUNK}
print("GPU",torch.cuda.get_device_name(),"Sequence: FULL FIRST → DELTA SECOND")


In [ ]:
#@title 2. Three official WikiText splits: TRAIN, DEV, sealed HOLDOUT
# Unlike V2, validation/test are not slices of the training split.
DATA_SOURCE="wikitext-2-raw-v1"
def clean_texts(split,needed):
    ds=load_dataset("Salesforce/wikitext","wikitext-2-raw-v1",split=split)
    unique=[];seen=set()
    for row in ds:
        value=" ".join(str(row["text"]).split())
        if 80<=len(value)<=850 and not value.startswith("=") and value not in seen:
            unique.append(value);seen.add(value)
    if len(unique)<needed:
        raise RuntimeError(f"Split {split} has only {len(unique)} eligible rows; need {needed}")
    rng=random.Random(SEED+{"train":11,"validation":23,"test":41}[split])
    rng.shuffle(unique)
    return unique[:needed]
try:
    train_text=clean_texts("train",TRAIN_EXAMPLES)
    dev_text=clean_texts("validation",DEV_EXAMPLES)
    holdout_text=clean_texts("test",HOLDOUT_EXAMPLES)
except Exception as exc:
    print("Official data unavailable; deterministic fallback for debugging ONLY:",repr(exc))
    DATA_SOURCE="synthetic_fallback_not_valid_for_publication"
    subjects=["A scientist","A library","An engineer","A hospital","A teacher",
              "A spacecraft","A mathematician","An orchestra","A historian","A botanist"]
    verbs=["described","studied","measured","compared","discussed",
           "reconsidered","observed","evaluated","investigated"]
    suffixes=["a new method using controlled experiments and careful analysis.",
              "the importance of verification and repeating the measurement.",
              "several observations that challenged the previous explanation.",
              "the distinction between efficiency, accuracy and usefulness.",
              "how practical design choices affected the final result."]
    needed=TRAIN_EXAMPLES+DEV_EXAMPLES+HOLDOUT_EXAMPLES
    unique=[(f"{subjects[i%len(subjects)]} "
             f"{verbs[(i//len(subjects))%len(verbs)]} "
             f"{suffixes[(i//(len(subjects)*len(verbs)))%len(suffixes)]} "
             f"Deterministic synthetic debug record {i}.")
            for i in range(needed)]
    print("WARNING: synthetic data is NOT a quality benchmark and HF upload will be blocked.")
    random.Random(SEED).shuffle(unique)
    train_text=unique[:TRAIN_EXAMPLES]
    dev_text=unique[TRAIN_EXAMPLES:TRAIN_EXAMPLES+DEV_EXAMPLES]
    holdout_text=unique[TRAIN_EXAMPLES+DEV_EXAMPLES:
                        TRAIN_EXAMPLES+DEV_EXAMPLES+HOLDOUT_EXAMPLES]
assert not set(train_text)&set(dev_text)
assert not set(train_text)&set(holdout_text)
assert not set(dev_text)&set(holdout_text)
print(DATA_SOURCE,"official split sizes",len(train_text),len(dev_text),len(holdout_text))



In [ ]:
#@title 3. Official Qwen3.5 text-only teacher; cache train and DEV logits once
teacher,tokenizer,source_root=wa.load_official_text(MODEL_ID,dtype=torch.float16)
teacher.to(device).eval().requires_grad_(False)
original_config=teacher.config.to_dict()

def make_record(text):
    ids=tokenizer(text,add_special_tokens=False)["input_ids"][:MAX_TOKENS+1]
    if len(ids)<8:raise ValueError("Text too short")
    return {"input_ids":torch.tensor([ids[:-1]],dtype=torch.long),
            "gold_next":int(ids[-1]),"text":text,"length":len(ids)-1}
train_rows=[make_record(s) for s in train_text]
dev_rows=[make_record(s) for s in dev_text]
# Holdout records are NOT yet materialized or evaluated.
@torch.inference_mode()
def attach_reference(records):
    for index,rec in enumerate(records):
        x=rec["input_ids"].to(device)
        logits=teacher(input_ids=x,use_cache=False,logits_to_keep=1).logits[:, -1]
        rec["teacher_logits"]=logits.float().cpu().contiguous()
        if index%25==0:print("Reference logits",index,"/",len(records))
    return records
train_rows=attach_reference(train_rows)
dev_rows=attach_reference(dev_rows)

teacher.cpu();gc.collect();torch.cuda.empty_cache()
print('Teacher logits cached for TRAIN/DEV only; HOLDOUT untouched.')


In [ ]:
#@title 4. Verify FULL attention first; export full-only version
def frozen_digest(model):
    h=hashlib.sha256()
    for i,L in enumerate(model.model.layers):
        for field in ("mlp","input_layernorm","post_attention_layernorm"):
            for key,tensor in getattr(L,field).state_dict().items():
                h.update(f"{i}/{field}/{key}".encode())
                h.update(tensor.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()

@torch.inference_mode()
def assess(model,records):
    model.eval()
    total=matches=0
    divergence=student_loss=teacher_loss=0.0
    for rec in records:
        ids=rec["input_ids"].to(device)
        with torch.autocast("cuda",dtype=torch.float16):
            logits=model(input_ids=ids,use_cache=False,logits_to_keep=1).logits[:,-1].float()
        ref=rec["teacher_logits"].to(device).float()
        p=logits.softmax(-1);q=ref.softmax(-1);m=(p+q)/2
        div=0.5*((p*(p.clamp_min(1e-10).log()-m.clamp_min(1e-10).log())).sum()
                 +(q*(q.clamp_min(1e-10).log()-m.clamp_min(1e-10).log())).sum())
        divergence+=float(div)
        matches+=int(logits.argmax(-1).item()==ref.argmax(-1).item())
        gold=torch.tensor([rec["gold_next"]],device=device)
        student_loss+=float(F.cross_entropy(logits,gold))
        teacher_loss+=float(F.cross_entropy(ref,gold))
        total+=1
    return {"n":total,"teacher_agreement":matches/total,"mean_js":divergence/total,
            "student_nll":student_loss/total,"teacher_nll":teacher_loss/total,
            "ppl_ratio":math.exp(min(30,(student_loss-teacher_loss)/total))}

FROZEN_DIGEST=frozen_digest(teacher)
FULL_IDS=[i for i,L in enumerate(teacher.model.layers)
          if L.block_type=="full_attention"]
DELTA_IDS=[i for i,L in enumerate(teacher.model.layers)
           if L.block_type=="linear_attention"]
assert len(FULL_IDS)==6 and len(DELTA_IDS)==18

def create_manifest(model,variant,dev,stage_accepted=False):
    installed=wa.stage_indices(model)
    assert installed["full"]==FULL_IDS,installed
    assert frozen_digest(model)==FROZEN_DIGEST,"Pretrained FFNs or block norms changed"
    return {
        "architecture":"Qwen3.5_0.8B_V4_FullFirst_ProgressiveDelta",
        "variant":variant,"source_model":MODEL_ID,"dtype":"float16",
        "attention_settings":ATTENTION_SETTINGS,
        "replaced_full_indices":installed["full"],
        "replaced_delta_indices":installed["delta"],
        "native_delta_layers_remaining":18-len(installed["delta"]),
        "text_only":True,"requires_use_cache_false":True,"no_cenn":True,
        "frozen_ffn_norm_sha256":FROZEN_DIGEST,
        "dev":dev,"stage_accepted":bool(stage_accepted),
        "dataset":DATA_SOURCE,"created_utc":datetime.now(timezone.utc).isoformat()
    }

def save_model_variant(model,folder,manifest):
    folder.mkdir(parents=True,exist_ok=True)
    model.half()
    wa.save_custom(model,tokenizer,folder,manifest)
    (folder/"README.md").write_text(
        f"# Qwen3.5 V4 — {manifest['variant']}\n\n"
        f"Full replaced {len(manifest['replaced_full_indices'])}; "
        f"Delta replaced {len(manifest['replaced_delta_indices'])}. "
        "Research only; text-only; use_cache=False. "
        "Load using tinycenn_lm.qwen35_window_anchor_v4.load_custom. "
        "Diagnostic-only variants are not qualified for deployment.\n")
    (folder/"requirements.txt").write_text(
        "torch>=2.4\nsafetensors>=0.5\n"
        "git+https://github.com/huggingface/transformers.git@main\n"
        "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@a17f294d0144c2a09c41ace8b9fbea6108af4eaa\n")
    assert (folder/"model.safetensors").is_file()
    model.to(device).eval()

full_model=copy.deepcopy(teacher)
wa.install_selected(full_model,full_indices=FULL_IDS,**ATTENTION_SETTINGS)
full_model.to(device).eval().requires_grad_(False)
assert wa.stage_indices(full_model)=={"full":FULL_IDS,"delta":[]}
assert frozen_digest(full_model)==FROZEN_DIGEST
FULL_DEV=assess(full_model,dev_rows)
FULL_PASS=bool(FULL_DEV["teacher_agreement"]>=0.99
               and FULL_DEV["mean_js"]<=0.002)
FULL_MANIFEST=create_manifest(full_model,"full6_only",FULL_DEV,FULL_PASS)
save_model_variant(full_model,FULL_EXPORT,FULL_MANIFEST)
(REPORTS/"full_only_dev.json").write_text(json.dumps(
    {"passes_full_stage_gate":FULL_PASS,"manifest":FULL_MANIFEST},indent=2))
print("Full attention first:",FULL_IDS)
print("All 18 native DeltaNet layers RETAINED.")
print("FULL-ONLY DEV",FULL_DEV,"PASS:",FULL_PASS)
print("Full-only standalone model saved:",FULL_EXPORT)
@torch.inference_mode()
def collect_probe_logits(model):
    model.eval()
    outputs=[]
    for rec in dev_rows[:4]:
        with torch.autocast("cuda",dtype=torch.float16):
            z=model(input_ids=rec["input_ids"].to(device),
                    use_cache=False,logits_to_keep=1).logits[:,-1]
        outputs.append(z.float().cpu().contiguous())
    return outputs
FULL_PREEXPORT_PROBE=collect_probe_logits(full_model)
DELTA_PREEXPORT_PROBE=None


In [ ]:
#@title 5. Rank all 18 potential Delta replacements by TRAIN-only damage
# All 6 full mixers are already replaced; native DeltaNet is preserved
# except for the ONE temporarily substituted module being measured.
DELTA_RANKING=[]
if FULL_PASS:
    subset=train_rows[:RANKING_EXAMPLES]
    reference_train=assess(full_model,subset)
    for index in DELTA_IDS:
        layer=full_model.model.layers[index]
        native=layer.linear_attn
        assert not isinstance(native,wa.WindowAnchorDeltaV2)
        candidate=wa.WindowAnchorDeltaV2(
            native,DELTA_WINDOW,ANCHORS,CHUNK).to(device).eval()
        candidate.requires_grad_(False)
        layer.linear_attn=candidate
        try:
            result=assess(full_model,subset)
            DELTA_RANKING.append({
                "layer":index,"teacher_agreement":result["teacher_agreement"],
                "mean_js":result["mean_js"],"ppl_ratio":result["ppl_ratio"],
                "top1_agreement_drop":
                   reference_train["teacher_agreement"]-result["teacher_agreement"]})
        finally:
            layer.linear_attn=native
            del candidate
            gc.collect()
        print("Single Delta trial L",index,
              "agreement",round(result["teacher_agreement"],4),
              "JS",round(result["mean_js"],5))
    DELTA_RANKING.sort(key=lambda x:(
        -x["teacher_agreement"],x["mean_js"],x["ppl_ratio"]))
    RANKED_DELTA=[x["layer"] for x in DELTA_RANKING]
else:
    print("FULL-6 fidelity gate failed: progressive Delta training blocked.")
    RANKED_DELTA=[]
(REPORTS/"single_delta_rankings.json").write_text(json.dumps({
    "full_only_passed":FULL_PASS,"teacher_train_baseline":
         reference_train if FULL_PASS else None,
    "train_probe_count":RANKING_EXAMPLES,"ranked_layers":DELTA_RANKING},indent=2))
print("Candidate Delta order (least damaging first):",RANKED_DELTA)


In [ ]:
#@title 6. Train progressive Delta replacements: 1 → 3 → 6 → 12 → 18, with rollback
# DeltaNet remains original in all unselected layers. A failed group restores
# its exact native modules, without mutating any previously accepted layer.
class DeltaLoRA(nn.Module):
    def __init__(self,base,rank=LORA_RANK):
        super().__init__()
        assert isinstance(base,nn.Linear)
        self.base=base.requires_grad_(False)
        dev=base.weight.device
        self.A=nn.Parameter(torch.empty(rank,base.in_features,
                             device=dev,dtype=torch.float32))
        self.B=nn.Parameter(torch.zeros(base.out_features,rank,
                             device=dev,dtype=torch.float32))
        nn.init.normal_(self.A,std=0.01)
        self.scale=1.0/rank
    def forward(self,x):
        return self.base(x)+F.linear(F.linear(x,self.A),self.B)*self.scale
    @torch.no_grad()
    def merge(self):
        delta=(self.B@self.A)*self.scale
        self.base.weight.add_(delta.to(dtype=self.base.weight.dtype))
        return self.base

def score(m):
    return (round(m["teacher_agreement"]
                  -0.4*max(0,m["ppl_ratio"]-1)-0.7*m["mean_js"],8),
            m["teacher_agreement"],-m["mean_js"])

def stage_train(new_layer_indices,stage_number):
    modules=[full_model.model.layers[i].linear_attn for i in new_layer_indices]
    trainable=[]
    wrapper_map=[]
    for module in modules:
        assert isinstance(module,wa.WindowAnchorDeltaV2)
        module.requires_grad_(False)
        for key in ("in_proj_qkv","in_proj_b","in_proj_z","out_proj"):
            lora=DeltaLoRA(getattr(module,key))
            setattr(module,key,lora)
            wrapper_map.append((module,key,lora))
            trainable.extend((lora.A,lora.B))
        for p in (module.log_temperature,module.log_decay,
                  module.direct_mix_logit,module.output_gain,
                  module.output_bias):
            p.requires_grad_(True)
            trainable.append(p)
        # GradScaler requires FP32 parameters when unscaling gradients.
        module.norm.float().requires_grad_(True)
        trainable.extend([p for p in module.norm.parameters() if p.requires_grad])
    assert trainable and all(p.dtype==torch.float32 for p in trainable)
    optim=torch.optim.AdamW(trainable,lr=LR_START,weight_decay=0)
    scaler=torch.amp.GradScaler("cuda")
    baseline=assess(full_model,dev_rows)
    best_metric=baseline
    best_params=[p.detach().cpu().clone() for p in trainable]
    best_step=0
    train_order=list(range(len(train_rows)))
    rng=random.Random(SEED+stage_number*307)
    rng.shuffle(train_order)
    pos=0
    log=[{"step":0,"dev":baseline}]
    for step in range(1,DELTA_STEPS+1):
        progress=(step-1)/max(1,DELTA_STEPS-1)
        lr=LR_START*(0.15+0.85*(0.5+0.5*math.cos(math.pi*progress)))
        optim.param_groups[0]["lr"]=lr
        optim.zero_grad(set_to_none=True)
        losses=[]
        for _ in range(GRAD_ACCUM):
            row=train_rows[train_order[pos]]
            pos=(pos+1)%len(train_order)
            if pos==0:rng.shuffle(train_order)
            ids=row["input_ids"].to(device)
            target=row["teacher_logits"].to(device).float()
            gold=torch.tensor([row["gold_next"]],device=device)
            with torch.autocast("cuda",dtype=torch.float16):
                pred=full_model(input_ids=ids,use_cache=False,
                                logits_to_keep=1).logits[:,-1].float()
                tau=2.0
                kd=F.kl_div(F.log_softmax(pred/tau,dim=-1),
                            F.softmax(target/tau,dim=-1),
                            reduction="batchmean")*tau*tau
                loss=kd+0.03*F.cross_entropy(pred,gold)
            if not bool(torch.isfinite(loss).item()):
                raise FloatingPointError(f"Stage {stage_number}, step {step}: nonfinite loss")
            scaler.scale(loss/GRAD_ACCUM).backward()
            losses.append(float(loss.detach()))
        scaler.unscale_(optim)
        norm=torch.nn.utils.clip_grad_norm_(trainable,0.6)
        if bool(torch.isfinite(norm).item()):
            scaler.step(optim)
        else:
            print("Skipped nonfinite gradient, stage",stage_number,"step",step)
        scaler.update()
        if step%EVAL_EVERY==0 or step==1 or step==DELTA_STEPS:
            dev=assess(full_model,dev_rows)
            if score(dev)>score(best_metric):
                best_metric=dev
                best_step=step
                best_params=[p.detach().cpu().clone() for p in trainable]
            log.append({"step":step,"loss":float(np.mean(losses)),
                        "dev":dev,"best_step":best_step})
            print(f"Stage {stage_number} step {step}/{DELTA_STEPS}: "
                  f"agree={dev['teacher_agreement']:.2%}, JS={dev['mean_js']:.5f}, "
                  f"best={best_step}")
    with torch.no_grad():
        for p,checkpoint in zip(trainable,best_params):
            p.copy_(checkpoint.to(device=p.device,dtype=p.dtype))
    for module,name,lora in wrapper_map:
        assert isinstance(getattr(module,name),DeltaLoRA)
        setattr(module,name,lora.merge())
    for module in modules:
        module.requires_grad_(False)
        module.half()
    del optim,scaler,trainable,best_params
    gc.collect();torch.cuda.empty_cache()
    return {"best_step":best_step,"history":log,
            "best_dev":best_metric,"final_dev":assess(full_model,dev_rows)}

STAGE_BOUNDARIES=(0,1,3,6,12,18)
STAGE_RECORDS=[]
ACCEPTED_DELTA=[]
CURRENT_DEV=FULL_DEV
DELTA_CANDIDATE_FOLDER=None
if FULL_PASS:
    for stage in range(1,len(STAGE_BOUNDARIES)):
        first,last=STAGE_BOUNDARIES[stage-1:stage+1]
        fresh=RANKED_DELTA[first:last]
        before={i:full_model.model.layers[i].linear_attn for i in fresh}
        # NEVER modify any of the previously accepted Delta layers.
        wa.install_selected(full_model,delta_indices=fresh,
                            **ATTENTION_SETTINGS)
        full_model.eval().requires_grad_(False)
        training=stage_train(fresh,stage)
        measured=training["final_dev"]
        passes=bool(
            measured["teacher_agreement"]>=max(0.95,
                   CURRENT_DEV["teacher_agreement"]-DEV_MAX_DROP)
            and measured["mean_js"]<=0.04
            and measured["ppl_ratio"]<=1.15)
        row={"stage":stage,"nominal_delta_target":last,"new_layers":fresh,
             "previous_dev":CURRENT_DEV,"candidate_dev":measured,
             "best_step":training["best_step"],"accepted":passes,
             "selected_delta_before":list(ACCEPTED_DELTA)}
        (REPORTS/f"stage{stage}_training.json").write_text(json.dumps(training,indent=2))
        # Ensure at least one genuine Delta version is preserved even when
        # all candidate groups degrade quality. It is DIAGNOSTIC ONLY.
        if not ACCEPTED_DELTA and DELTA_CANDIDATE_FOLDER is None:
            diagnostic=create_manifest(
                full_model,"rejected_delta_diagnostic",measured,False)
            save_model_variant(full_model,DIAGNOSTIC_EXPORT,diagnostic)
            DELTA_CANDIDATE_FOLDER=DIAGNOSTIC_EXPORT
            DELTA_PREEXPORT_PROBE=collect_probe_logits(full_model)
        if passes:
            ACCEPTED_DELTA.extend(fresh)
            ACCEPTED_DELTA.sort()
            CURRENT_DEV=measured
        else:
            for index,native in before.items():
                full_model.model.layers[index].linear_attn=native
            gc.collect();torch.cuda.empty_cache()
        row["selected_delta_after"]=list(ACCEPTED_DELTA)
        STAGE_RECORDS.append(row)
        assert frozen_digest(full_model)==FROZEN_DIGEST
        print(f"Stage {stage}: {'ACCEPTED' if passes else 'ROLLED BACK'}; "
              f"selected {len(ACCEPTED_DELTA)}/18 Delta layers")
    if ACCEPTED_DELTA:
        best_manifest=create_manifest(
            full_model,"progressive_delta_accepted",CURRENT_DEV,True)
        save_model_variant(full_model,DELTA_EXPORT,best_manifest)
        DELTA_CANDIDATE_FOLDER=DELTA_EXPORT
        DELTA_PREEXPORT_PROBE=collect_probe_logits(full_model)
else:
    print("No Delta substitutions attempted: full-only compatibility failed.")

DELTA_STAGE_PASSED=bool(ACCEPTED_DELTA)
(REPORTS/"progressive_delta_stages.json").write_text(json.dumps({
    "full_stage_passed":FULL_PASS,"stages":STAGE_RECORDS,
    "accepted_delta_layer_indices":ACCEPTED_DELTA,
    "candidate_path":str(DELTA_CANDIDATE_FOLDER) if DELTA_CANDIDATE_FOLDER else None,
    "delta_stage_accepted":DELTA_STAGE_PASSED},indent=2))
print("Final accepted Delta layers:",ACCEPTED_DELTA)
print("Selected Delta checkpoint folder:",DELTA_CANDIDATE_FOLDER)


In [ ]:
#@title 7. Sealed HOLDOUT, true FP16 prefill latency and strict standalone reload
# The HOLDOUT is first accessed HERE, after selection and all rollbacks.
full_model.cpu()
del full_model
gc.collect();torch.cuda.empty_cache()
teacher.to(device).eval().requires_grad_(False)
holdout_rows=[make_record(text) for text in holdout_text]
assert len(holdout_rows)==HOLDOUT_EXAMPLES
holdout_rows=attach_reference(holdout_rows)

@torch.inference_mode()
def prefill_ms(model,ids,repeats=7):
    model.eval()
    ids=ids.to(device)
    for _ in range(2):
        with torch.autocast("cuda",dtype=torch.float16):
            model(input_ids=ids,use_cache=False,logits_to_keep=1)
    torch.cuda.synchronize()
    durations=[]
    for _ in range(repeats):
        a=torch.cuda.Event(enable_timing=True)
        b=torch.cuda.Event(enable_timing=True)
        a.record()
        with torch.autocast("cuda",dtype=torch.float16):
            model(input_ids=ids,use_cache=False,logits_to_keep=1)
        b.record();b.synchronize()
        durations.append(float(a.elapsed_time(b)))
    return float(np.median(durations))

BENCH_LENGTHS=(64,128,192)
sample=holdout_rows[0]["input_ids"].to(device)
teacher_speed={}
for length in BENCH_LENGTHS:
    ids=sample.repeat(1,math.ceil(length/sample.shape[-1]))[:,:length]
    teacher_speed[length]=prefill_ms(teacher,ids)
teacher.cpu();gc.collect();torch.cuda.empty_cache()
print("Native teacher FP16 prefill:",teacher_speed)

def inspect_export(folder,reference_logits,label):
    model,tok,layout=wa.load_custom(folder,device=device)
    try:
        metadata=json.loads((folder/"window_anchor_manifest.json").read_text())
        expected={"full":metadata["replaced_full_indices"],
                  "delta":metadata["replaced_delta_indices"]}
        assert layout==expected,("Wrong restored mixer layout",label,layout)
        assert frozen_digest(model)==FROZEN_DIGEST,("Frozen weights mismatch",label)
        holdout=assess(model,holdout_rows)
        timing=[]
        for length in BENCH_LENGTHS:
            inp=sample.repeat(1,math.ceil(length/sample.shape[-1]))[:,:length]
            student_time=prefill_ms(model,inp)
            timing.append({
                "length":length,"teacher_ms":teacher_speed[length],
                "student_ms":student_time,
                "speedup":teacher_speed[length]/max(student_time,1e-9),
                "batch":1,"use_cache":False})
        max_abs=0.;top_match=0;js_total=0.
        with torch.inference_mode(),torch.autocast("cuda",dtype=torch.float16):
            for rec,ref in zip(dev_rows[:4],reference_logits):
                x=rec["input_ids"].to(device)
                logits=model(input_ids=x,use_cache=False,
                             logits_to_keep=1).logits[:,-1].float()
                ref=ref.to(device).float()
                max_abs=max(max_abs,float((logits-ref).abs().max()))
                top_match+=int(logits.argmax(-1).item()==ref.argmax(-1).item())
                p=logits.softmax(-1);q=ref.softmax(-1);middle=(p+q)/2
                js_total+=float(0.5*((p*(p.clamp_min(1e-10).log()-middle.clamp_min(1e-10).log())).sum()
                                     +(q*(q.clamp_min(1e-10).log()-middle.clamp_min(1e-10).log())).sum()))
        parity={"probe_count":len(reference_logits),"max_abs_logit":max_abs,
                "top1_matches":top_match,"mean_js":js_total/len(reference_logits),
                "strict_loader_passed":True}
        parity["passed"]=(top_match==len(reference_logits)
                          and max_abs<=0.05 and parity["mean_js"]<=1e-4)
        result={"variant":label,"holdout":holdout,
                "timing":timing,"median_speedup":
                    float(np.median([t["speedup"] for t in timing])),
                "reload":parity,"layout":layout}
        print("Variant:",label,"HOLDOUT:",holdout,
              "median speedup:",result["median_speedup"],"reload:",parity)
        return result
    finally:
        del model
        gc.collect();torch.cuda.empty_cache()

FULL_RESULT=inspect_export(FULL_EXPORT,FULL_PREEXPORT_PROBE,"full6_only")
DELTA_RESULT=None
if DELTA_CANDIDATE_FOLDER is not None:
    assert DELTA_PREEXPORT_PROBE is not None
    DELTA_RESULT=inspect_export(
        DELTA_CANDIDATE_FOLDER,DELTA_PREEXPORT_PROBE,
        "progressive_delta" if DELTA_STAGE_PASSED else "rejected_delta_diagnostic")
(REPORTS/"sealed_holdout_and_speed.json").write_text(json.dumps({
    "full_only":FULL_RESULT,"delta_variant":DELTA_RESULT,
    "teacher_no_cache_ms":teacher_speed},indent=2))


In [ ]:
#@title 8. Compare teacher/full-only/Delta chat and apply separate quality gates
CHAT_PROMPTS=[
    "Explain in two sentences why the sky appears blue.",
    "Calculate 17 multiplied by 23. Answer with the number.",
    "Write a friendly greeting in German.",
    "Explain the difference between recurrent and full attention.",
    "Name two safety precautions when operating a small drone."
]
@torch.inference_mode()
def generate_reply(model,prompt,steps=24):
    msgs=[{"role":"user","content":prompt}]
    try:
        try:
            inputs=tokenizer.apply_chat_template(msgs,tokenize=True,
                add_generation_prompt=True,return_tensors="pt",enable_thinking=False)
        except TypeError:
            inputs=tokenizer.apply_chat_template(msgs,tokenize=True,
                add_generation_prompt=True,return_tensors="pt")
    except Exception:
        inputs=tokenizer("User: "+prompt+"\nAssistant:",
                         return_tensors="pt")["input_ids"]
    if isinstance(inputs,dict):inputs=inputs["input_ids"]
    if hasattr(inputs,"input_ids"):inputs=inputs.input_ids
    if not isinstance(inputs,torch.Tensor):
        inputs=torch.as_tensor(inputs,dtype=torch.long)
    if inputs.ndim==1:inputs=inputs[None]
    assert inputs.ndim==2
    inputs=inputs.to(device)
    beginning=inputs.shape[1]
    for _ in range(steps):
        with torch.autocast("cuda",dtype=torch.float16):
            next_logits=model(input_ids=inputs,use_cache=False,
                              logits_to_keep=1).logits[:,-1]
        token=next_logits.argmax(-1,keepdim=True)
        inputs=torch.cat((inputs,token),dim=1)
        if int(token[0,0])==tokenizer.eos_token_id:break
    return tokenizer.decode(inputs[0,beginning:],skip_special_tokens=True).strip()

def plausible_answer(answer):
    tokens=re.findall(r"\w+",answer or "",flags=re.UNICODE)
    if not tokens:return False
    if len(tokens)==1:return any(ch.isdigit() for ch in tokens[0])
    if len(tokens)<3:return False
    if len(set(w.lower() for w in tokens))/len(tokens)<0.5:return False
    return not any(tokens[i].lower()==tokens[i+1].lower()==tokens[i+2].lower()
                   for i in range(len(tokens)-2))

def chat_trial(model,label):
    rows=[]
    for prompt in CHAT_PROMPTS:
        try:
            answer=generate_reply(model,prompt)
            row={"prompt":prompt,"answer":answer,
                 "appears_usable":plausible_answer(answer)}
        except Exception as exc:
            row={"prompt":prompt,"answer":None,"appears_usable":False,
                 "error":type(exc).__name__+": "+str(exc)}
        rows.append(row)
        print(label,"USER:",prompt,"RESPONSE:",row["answer"])
    return rows

def chat_from_export(folder,label):
    model,_,_=wa.load_custom(folder,device=device)
    try:return chat_trial(model,label)
    finally:
        del model
        gc.collect();torch.cuda.empty_cache()

teacher.to(device).eval()
TEACHER_CHAT=chat_trial(teacher,"TEACHER")
teacher.cpu();gc.collect();torch.cuda.empty_cache()
FULL_CHAT=chat_from_export(FULL_EXPORT,"FULL_ONLY")
DELTA_CHAT=(chat_from_export(DELTA_CANDIDATE_FOLDER,"DELTA")
            if DELTA_CANDIDATE_FOLDER is not None else [])
FULL_CHAT_OK=sum(int(x["appears_usable"]) for x in FULL_CHAT)
DELTA_CHAT_OK=sum(int(x["appears_usable"]) for x in DELTA_CHAT)
FULL_QUALIFIED=bool(
    FULL_PASS and FULL_RESULT["holdout"]["teacher_agreement"]>=0.995
    and FULL_RESULT["holdout"]["mean_js"]<=0.002
    and FULL_RESULT["median_speedup"]>=0.95
    and FULL_RESULT["reload"]["passed"] and FULL_CHAT_OK>=4
    and DATA_SOURCE=="wikitext-2-raw-v1")
DELTA_QUALIFIED=bool(
    DELTA_STAGE_PASSED and DELTA_RESULT is not None
    and DELTA_RESULT["holdout"]["teacher_agreement"]>=0.95
    and DELTA_RESULT["holdout"]["mean_js"]<=0.04
    and DELTA_RESULT["holdout"]["ppl_ratio"]<=1.15
    and DELTA_RESULT["median_speedup"]>=1.0
    and DELTA_RESULT["reload"]["passed"] and DELTA_CHAT_OK>=4
    and DATA_SOURCE=="wikitext-2-raw-v1")
for folder,qualified,result,chats in (
        (FULL_EXPORT,FULL_QUALIFIED,FULL_RESULT,FULL_CHAT),
        (DELTA_CANDIDATE_FOLDER,DELTA_QUALIFIED,DELTA_RESULT,DELTA_CHAT)):
    if folder is None:continue
    manifest_file=folder/"window_anchor_manifest.json"
    manifest=json.loads(manifest_file.read_text())
    manifest["qualified"]=bool(qualified)
    manifest["holdout"]=result["holdout"]
    manifest["benchmark"]=result["timing"]
    manifest["reload"]=result["reload"]
    manifest["appears_usable_chats"]=sum(int(z["appears_usable"]) for z in chats)
    manifest_file.write_text(json.dumps(manifest,indent=2))
(REPORTS/"chat_comparison.json").write_text(json.dumps({
    "teacher":TEACHER_CHAT,"full_only":FULL_CHAT,
    "progressive_delta":DELTA_CHAT,"heuristic_only":True},indent=2))
(REPORTS/"final_report.json").write_text(json.dumps({
    "full_only":{"metrics":FULL_RESULT,"qualified":FULL_QUALIFIED},
    "delta":{"metrics":DELTA_RESULT,"qualified":DELTA_QUALIFIED,
             "accepted_layers":ACCEPTED_DELTA,
             "diagnostic_only":not DELTA_STAGE_PASSED},
    "frozen_sha256":FROZEN_DIGEST,"dataset":DATA_SOURCE,
    "created_utc":datetime.now(timezone.utc).isoformat()},indent=2))
print("HF QUALITY GATES: FULL",FULL_QUALIFIED,"DELTA",DELTA_QUALIFIED)


In [ ]:
#@title 9. Publish FULL or DELTA model to Hugging Face only if independently qualified
HF_RESULTS={
    "full_only":{"qualified":FULL_QUALIFIED,"uploaded":False},
    "progressive_delta":{"qualified":DELTA_QUALIFIED,"uploaded":False}
}
if UPLOAD_IF_QUALIFIED and (FULL_QUALIFIED or DELTA_QUALIFIED):
    try:
        token=get_token()
        if not token:
            login()
            token=get_token()
        assert token,"Hugging Face write token required"
        api=HfApi(token=token)
        username=api.whoami(token=token)["name"]
        for name,folder,approved,repo_suffix in (
            ("full_only",FULL_EXPORT,FULL_QUALIFIED,FULL_HF_NAME),
            ("progressive_delta",DELTA_CANDIDATE_FOLDER,
             DELTA_QUALIFIED,DELTA_HF_NAME)):
            if not approved or folder is None:continue
            repository=f"{username}/{repo_suffix}"
            try:
                api.create_repo(repo_id=repository,repo_type="model",
                                private=HF_PRIVATE,exist_ok=True)
                api.upload_folder(repo_id=repository,repo_type="model",
                                  folder_path=str(folder),
                                  commit_message="V4 staged Qwen: full-first then DEV-qualified Delta")
                files=set(api.list_repo_files(repo_id=repository,repo_type="model"))
                required={"model.safetensors","window_anchor_manifest.json",
                          "qwen35_window_anchor_v4.py","config.json","README.md"}
                assert required<=files,sorted(required-files)
                HF_RESULTS[name].update({
                    "uploaded":True,"repo_id":repository,
                    "url":"https://huggingface.co/"+repository})
                print("Published qualified",name,HF_RESULTS[name]["url"])
            except Exception as exc:
                HF_RESULTS[name]["error"]=repr(exc)
                print("HF upload failed for",name,repr(exc))
    except Exception as exc:
        HF_RESULTS["auth_error"]=repr(exc)
        print("Hugging Face login unavailable:",repr(exc))
else:
    print("HF upload disabled or neither candidate passed quality gates.")
(REPORTS/"hf_upload_status.json").write_text(json.dumps(HF_RESULTS,indent=2))


In [ ]:
#@title 10. Save FINAL statistics ZIP and both FP16 model ZIPs; browser download
from IPython.display import display,FileLink
STATS_ZIP=BASE/"Qwen35_V4_FullFirst_Delta_Statistics.zip"
FULL_ZIP=BASE/"Qwen35_V4_Full6_Only_Model.zip"
DELTA_ZIP=BASE/("Qwen35_V4_Progressive_Delta_Model.zip"
                if DELTA_STAGE_PASSED else "Qwen35_V4_Delta_Diagnostic_Only.zip")
sources=sorted(p for p in REPORTS.rglob("*") if p.is_file())
assert (REPORTS/"final_report.json") in sources
assert (REPORTS/"hf_upload_status.json") in sources
with zipfile.ZipFile(STATS_ZIP,"w",compression=zipfile.ZIP_DEFLATED) as z:
    for p in sources:z.write(p,"statistics/"+p.relative_to(REPORTS).as_posix())

def create_model_zip(folder,zip_path):
    assert (folder/"model.safetensors").is_file()
    with zipfile.ZipFile(zip_path,"w",allowZip64=True) as z:
        for file in sorted(p for p in folder.rglob("*") if p.is_file()):
            z.write(file,"model/"+file.relative_to(folder).as_posix(),
                    compress_type=(zipfile.ZIP_STORED
                                   if file.suffix==".safetensors"
                                   else zipfile.ZIP_DEFLATED))
        for file in sources:
            z.write(file,"statistics/"+file.relative_to(REPORTS).as_posix(),
                    compress_type=zipfile.ZIP_DEFLATED)
    with zipfile.ZipFile(zip_path) as z:
        assert "model/model.safetensors" in z.namelist()
    print("Model ZIP ready:",zip_path,"GiB",round(zip_path.stat().st_size/2**30,3))

create_model_zip(FULL_EXPORT,FULL_ZIP)
if DELTA_CANDIDATE_FOLDER is not None:
    create_model_zip(DELTA_CANDIDATE_FOLDER,DELTA_ZIP)
with zipfile.ZipFile(STATS_ZIP) as z:
    assert z.testzip() is None
print("Statistics ZIP verified:",STATS_ZIP)
display(FileLink(str(STATS_ZIP)))
display(FileLink(str(FULL_ZIP)))
if DELTA_CANDIDATE_FOLDER is not None:display(FileLink(str(DELTA_ZIP)))
try:
    from google.colab import files
    if AUTO_DOWNLOAD_STATS:files.download(str(STATS_ZIP))
    if AUTO_DOWNLOAD_MODELS:
        files.download(str(FULL_ZIP))
        if DELTA_CANDIDATE_FOLDER is not None:files.download(str(DELTA_ZIP))
except Exception as exc:
    print("Browser download unavailable:",repr(exc))
    print("Use FileLinks above or the Colab Files sidebar.")
print("COMPLETE. Native Delta kept in Full-only:",18,
      "| Delta replaced in accepted candidate:",len(ACCEPTED_DELTA))
print("Hugging Face:",HF_RESULTS)
